In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, random_split
from torchtext.data.utils import get_tokenizer
from collections import defaultdict
from torch.nn.utils.rnn import pad_sequence
from collections import Counter

dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data  = dataset["test"]

print(Counter(train_data["label"]))  
print(Counter(test_data["label"])) 


/home/aayam/micromamba/envs/RNN_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/aayam/micromamba/envs/RNN_env/lib/python3.11/site-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


Counter({0: 12500, 1: 12500})
Counter({0: 12500, 1: 12500})


In [2]:
tokenizer= get_tokenizer("basic_english")

vocab= defaultdict(lambda: 0)

counter=1

for text in dataset["train"]["text"]:
    for token in tokenizer(text):
        if token not in vocab:
            vocab[token]= counter
            counter+=1

vocab_size= len(vocab) + 1


def text_pipeline(text, vocab=vocab):
    return [vocab[token] for token in tokenizer(text)]
    

In [3]:
class SentimentClassifier(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,output_dim):
        super().__init__()

        self.embedding= nn.Embedding(vocab_size, embedding_dim)
        self.lstm= nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc= nn.Linear(hidden_dim, output_dim)

    
    def forward(self, text):

        embedding= self.embedding(text)
        output, (h_n,c_n)= self.lstm(embedding)
        last_hidden_state= h_n[-1]
        out= self.fc(last_hidden_state)

        return out 

device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model= SentimentClassifier(vocab_size=vocab_size,embedding_dim=100, hidden_dim=256, output_dim=1)
model= model.to(device)


In [6]:
def collate(batch):

    text_list, label_list= [],[]
    max_len= 256

    for item in batch:

        label_list.append(item['label'])
        processed_text= torch.tensor(text_pipeline(item["text"],vocab)[:max_len], dtype=torch.int64)
        text_list.append(processed_text)

    text_list= pad_sequence(text_list, batch_first=True, padding_value=0)
    label_list= torch.tensor(label_list)

    return label_list, text_list

train_size= int(0.9 * len(train_data))
val_size= len(train_data)- train_size

train_set, val_set= random_split(train_data,[train_size,val_size])

data_loader= DataLoader(train_set, batch_size=64,shuffle=True, collate_fn=collate)
test_loader= DataLoader(test_data,batch_size=64,shuffle=False, collate_fn=collate)
val_loader= DataLoader(val_set,batch_size=64,shuffle=False, collate_fn=collate)

In [7]:
optimizer= torch.optim.AdamW(model.parameters(), lr=0.001)
criterion= nn.BCEWithLogitsLoss()

num_epochs=10

best_loss= float("inf")

for epoch in range(num_epochs):

    model.train()
    running_loss=0

    for batch_idx, (label,text) in enumerate(data_loader):

        label= label.float().unsqueeze(1).to(device)
        text=  text.to(device)

        output= model(text)
        loss= criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss+=loss.item()

        if batch_idx % 100 ==0:
            print(f"epoch {epoch+1}/{num_epochs} | batch {batch_idx+1}/{len(data_loader)} | loss {loss.item()}")
   
    average_loss= running_loss/len(data_loader)
    print(f"average loss of epoch {epoch+1}/{num_epochs} | : {average_loss}")
    
    with torch.no_grad():
        model.eval()
        running_val_loss=0

        for label,text in val_loader:

            label= label.float().unsqueeze(1).to(device)
            text= text.to(device)

            output= model(text)
            loss= criterion(output,label)

            running_val_loss+= loss.item()

    average_val_loss= running_val_loss/ len(val_loader)
    print(f"average validation loss of epoch {epoch+1}/{num_epochs} | : {average_val_loss}")

    if average_val_loss < best_loss:
        best_loss= average_val_loss
        torch.save(model.state_dict(), "best_parameters.pth")


epoch 1/10 | batch 1/352 | loss 0.6977433562278748
epoch 1/10 | batch 101/352 | loss 0.7042426466941833
epoch 1/10 | batch 201/352 | loss 0.6982671022415161
epoch 1/10 | batch 301/352 | loss 0.7026448249816895
average loss of epoch 1/10 | : 0.6928798918696967
average validation loss of epoch 1/10 | : 0.6922227770090104
epoch 2/10 | batch 1/352 | loss 0.6870237588882446
epoch 2/10 | batch 101/352 | loss 0.7019939422607422
epoch 2/10 | batch 201/352 | loss 0.6783524751663208
epoch 2/10 | batch 301/352 | loss 0.6633009314537048
average loss of epoch 2/10 | : 0.6848271191120148
average validation loss of epoch 2/10 | : 0.6848092198371887
epoch 3/10 | batch 1/352 | loss 0.6774640083312988
epoch 3/10 | batch 101/352 | loss 0.7054104208946228
epoch 3/10 | batch 201/352 | loss 0.6498110294342041
epoch 3/10 | batch 301/352 | loss 0.7165011167526245
average loss of epoch 3/10 | : 0.6657722863284025
average validation loss of epoch 3/10 | : 0.6900024831295013
epoch 4/10 | batch 1/352 | loss 0.658

In [13]:
model.load_state_dict(torch.load("best_parameters.pth", map_location=device))
def predict_sentiment(text):
    
    with torch.no_grad():
        model.eval()

        text_tensor= torch.tensor(text_pipeline(text, vocab), dtype=torch.int64).unsqueeze(0).to(device)
        prediction= model(text_tensor)
        sentiment= 'positive' if prediction>0.5 else 'negative'

        return sentiment, prediction
    

reviews= [
    'The movie was really great',
    'I really loved the movie',
    'It was really bad',
    'it was okay at its best',
    'Its not bad, but I wouldnt watch it again',
    'I expected it to be terrible… and somehow it wasnt',
    'The acting was great, the plot was painful.'
    
]

for review in reviews:
    sentiment,prediction= predict_sentiment(review)

    print(f"review: {review}")
    print(f"sentiment: {sentiment}")


review: The movie was really great
sentiment: positive
review: I really loved the movie
sentiment: negative
review: It was really bad
sentiment: negative
review: it was okay at its best
sentiment: negative
review: Its not bad, but I wouldnt watch it again
sentiment: negative
review: I expected it to be terrible… and somehow it wasnt
sentiment: negative
review: The acting was great, the plot was painful.
sentiment: negative


In [14]:
total=0
correct=0
TrueP = 0
TrueN = 0
FalseP = 0
FalseN = 0

model.load_state_dict(torch.load("best_parameters.pth", map_location=device))

with torch.no_grad():
   model.eval()

   for label,text in test_loader:

      label= label.to(device).float()
      text=  text.to(device)

      output= model(text)

      probs= torch.sigmoid(output)
      preds= (probs>=0.5).float()

      total += label.size(0)
      correct += (preds.squeeze()==label).sum().item()

      TrueP += ((preds == 1) & (label == 1)).sum().item()
      TrueN += ((preds == 0) & (label == 0)).sum().item()
      FalseP += ((preds == 1) & (label == 0)).sum().item()
      FalseN += ((preds == 0) & (label == 1)).sum().item()

   conf_matrix = torch.tensor([[TrueP, FalseN],
                               [FalseP, TrueN]])
   accuracy= 100 * correct/total

print(accuracy)
print(conf_matrix)

80.924
tensor([[670420, 128620],
        [177828, 622172]])
